1. Logistic Regression       → fast baseline, interpretable, sets the bar
2. Decision Tree             → another interpretable baseline, no scaling needed
3. Random Forest             → almost always beats both of the above, still reasonably fast
4. SVM                       → try once scaling is solid, compare against Random Forest
5. Neural Network → Module 11+, likely overkill for this tabular dataset but 
                                 good for demonstrating you've covered that part of the course

In [100]:
import os
import joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, log_loss, classification_report, hinge_loss

In [3]:
df = pd.read_csv('../data/processed/df_cleaned.csv')
df

,is_tv_subscriber,is_movie_package_subscriber,subscription_age,bill_avg,remaining_contract,service_failure_count,download_avg,upload_avg,download_over_limit,churn,has_contract_info,has_active_contract
0,1,0,11.95,25,0.14,0,8.4,2.3,0,0,1,1
1,0,0,8.22,0,0.00,0,0.0,0.0,0,1,0,0
2,1,0,8.91,16,0.00,0,13.7,0.9,0,1,1,0
3,0,0,6.87,21,0.00,1,0.0,0.0,0,1,0,0
4,0,0,6.39,0,0.00,0,0.0,0.0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
72269,1,1,0.09,0,1.25,0,0.0,0.0,0,1,1,1
72270,1,0,0.06,1,1.63,0,0.8,0.0,0,1,1,1
72271,1,0,0.02,0,2.19,0,1.5,0.2,0,1,1,1
72272,0,0,0.01,0,0.72,0,0.0,0.0,0,1,1,1


In [11]:
X = df.drop(columns=['churn'])
y = df['churn']

In [14]:
X_train, X_test,y_train, y_test = train_test_split(X,y,test_size=0.2,stratify=y,random_state=19)

In [15]:
# Українська версія: Стандартизація оброблюваних даних
# English version: Standartization of processed data

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### 1. Модель логістичної регресії (Logistic Regression Model)

In [93]:
# Українська версія: Визначення типу моделі, параметрів та класифікатора.
# English version: Definition of the model type, parameters and classifier. 
model_lg = LogisticRegression(solver='saga',max_iter=5000,random_state=19)

parameters = [
    {
        'C': [0.01, 0.1, 1,10,100],
        'penalty': ['l1', 'l2'],
        'class_weight':['balanced', None]     
    },
]

clf = GridSearchCV(estimator=model_lg,param_grid=parameters)

In [94]:
# Train model
clf.fit(X_train_scaled, y_train)

# Predictions
y_pred_lg = clf.predict(X_test_scaled)

# Accuracy score
acc_lg = accuracy_score(y_pred_lg, y_test)

# Loss score
y_pred_proba_lg = clf.predict_proba(X_test_scaled)[:,1]
loss_lg = log_loss(y_test,y_pred_proba_lg)

In [95]:
final = classification_report(y_test, y_pred_lg)

print('Accuracy score of Logistic Regression: ', acc_lg)
print('Loss score of Logistic Regression: ', loss_lg)
print('\t\tFinal classification report:')
print(final)

Accuracy score of Logistic Regression:  0.9236250432376341
Loss score of Logistic Regression:  0.2364460376938783
		Final classification report:
              precision    recall  f1-score   support

           0       0.89      0.94      0.92      6445
           1       0.95      0.91      0.93      8010

    accuracy                           0.92     14455
   macro avg       0.92      0.93      0.92     14455
weighted avg       0.93      0.92      0.92     14455



In [96]:
best_params = clf.best_params_
best_estimator = clf.best_estimator_
print('Best parameters are: ', best_params)
print('Best logistic model is: ', best_estimator)

Best parameters are:  {'C': 10, 'class_weight': None, 'penalty': 'l2'}
Best logistic model is:  LogisticRegression(C=10, max_iter=5000, random_state=19, solver='saga')


In [99]:
logistic_models_location = '../models/logistic_regression'

os.makedirs(logistic_models_location, exist_ok=True)

model_path = os.path.join(logistic_models_location, 'best_logistic_regression_model.pkl')

joblib.dump(best_estimator,model_path)

['../models/logistic_regression/best_logistic_regression_model.pkl']

### 2. Модель «дерево рішень» (The Decision Tree Model)

In [104]:
# 1. Initialize the model
decision_tree_model = DecisionTreeClassifier(random_state=19)

# 2. Parameters definition
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [3, 5, 10, None],
    'min_samples_leaf': [1, 2, 4]
}
# 3. Classifier 
clf_dt = GridSearchCV(estimator=decision_tree_model,param_grid=param_grid,cv=10)

# 4. Training
clf_dt.fit(X_train_scaled, y_train)

# 5. Prediction
y_pred_dt = clf_dt.predict(X_test_scaled)

# 6. Calculate accuracy and other metrics
acc_dt = accuracy_score(y_test, y_pred_dt)
class_report_dt = classification_report(y_test, y_pred_dt)

In [106]:
acc_dt_rounded = round(acc_dt * 100, 2)
print(f'The accuracy score of the Decision Tree model is {acc_dt_rounded} ({acc_dt})')
# print('Loss score of Logistic Regression: ', loss_lg)
print('\t\tClassification report for Decision Tree model :')
print(class_report_dt)

The accuracy score of the Decision Tree model is 93.91 (0.9390522310619163)
		Classification report for Decision Tree model :
              precision    recall  f1-score   support

           0       0.92      0.95      0.93      6445
           1       0.96      0.93      0.94      8010

    accuracy                           0.94     14455
   macro avg       0.94      0.94      0.94     14455
weighted avg       0.94      0.94      0.94     14455



In [108]:
best_params_dt = clf_dt.best_params_
best_estimator_dt = clf_dt.best_estimator_
print('Best parameters are: ', best_params_dt)
print('Best logistic model is: ', best_estimator_dt)

Best parameters are:  {'criterion': 'entropy', 'max_depth': 10, 'min_samples_leaf': 4}
Best logistic model is:  DecisionTreeClassifier(criterion='entropy', max_depth=10, min_samples_leaf=4,
                       random_state=19)


If time allows, build a visual graphic

In [111]:
decision_tree_location = '../models/decision_tree'

os.makedirs(decision_tree_location, exist_ok=True)

model_path = os.path.join(decision_tree_location, 'best_decision_tree_model.pkl')

joblib.dump(best_estimator_dt,model_path)

['../models/decision_tree/best_decision_tree_model.pkl']